# Thermodynamic model antibody aggregation prediction

This is the only notebook a user needs to run the model.

Put in one antibody `.pdb` file, a folder of `.pdb` files, or a manifest CSV. The notebook returns a thermodynamic aggregation-risk score, a qualitative rank, and a pass/non-pass call.


## Model idea, in one paragraph

The model treats aggregation as a nucleation-like thermodynamic event. Antibody surface features are converted into an effective aggregation driving force \(\Delta\mu_{\mathrm{proxy}}\) and an effective interfacial penalty \(\gamma_{\mathrm{proxy}}\). These define a barrier

\[
\Delta G(n)=\gamma_{\mathrm{proxy}}n^{2/3}-n|\Delta\mu_{\mathrm{proxy}}|,
\]

and lower barriers are converted into higher aggregation-risk scores. The score is best used for ranking first; the pass/non-pass call is a secondary thresholded readout.


## Step 1: choose input

Edit this cell for your own antibody.

- `pdb_file`: one PDB file
- `pdb_directory`: every `.pdb` file in a folder
- `manifest`: CSV with columns `pdb_path, antibody_id, pdb_id, heavy_chain, light_chain`

If you know the heavy/light chain IDs, provide them. If not, leave them blank and the notebook will try to use the two largest protein chains.


In [ ]:
# ===== User input =====
INPUT_MODE = "pdb_file"
INPUT_PATH = "../examples/pdbs/5VH3.pdb"  # replace with your own PDB path

HEAVY_CHAIN = "H"
LIGHT_CHAIN = "L"

RUN_NAME = "thermodynamic_model_prediction"
MAX_CURVATURE_POINTS = 800
# ======================


## Step 2: load the model

Run this cell once. It imports the local package directly from this repository.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from abthermo_aggregation.pdb_surface import compute_surface_descriptors, infer_two_largest_protein_chains
from abthermo_aggregation.scoring import add_batch_scores, DEFAULT_THERMODYNAMIC_MODEL_THRESHOLD

OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Model loaded from:", SRC)
print("Output folder:", OUTPUT_DIR)
print("Pass/non-pass threshold:", DEFAULT_THERMODYNAMIC_MODEL_THRESHOLD)


## Step 3: prepare the PDB list

This turns your input into a simple table of PDB files to score.


In [ ]:
def resolve_path(path, base_dir):
    path = Path(str(path)).expanduser()
    if not path.is_absolute():
        path = base_dir / path
    return path.resolve()


def build_input_table(input_mode, input_path):
    input_path = resolve_path(input_path, NOTEBOOK_DIR)

    if input_mode == "pdb_file":
        return pd.DataFrame([{
            "pdb_path": str(input_path),
            "antibody_id": input_path.stem,
            "pdb_id": input_path.stem,
            "heavy_chain": HEAVY_CHAIN,
            "light_chain": LIGHT_CHAIN,
        }])

    if input_mode == "pdb_directory":
        pdb_files = sorted(input_path.glob("*.pdb"))
        if not pdb_files:
            raise ValueError(f"No .pdb files found in {input_path}")
        return pd.DataFrame([{
            "pdb_path": str(p),
            "antibody_id": p.stem,
            "pdb_id": p.stem,
            "heavy_chain": HEAVY_CHAIN,
            "light_chain": LIGHT_CHAIN,
        } for p in pdb_files])

    if input_mode == "manifest":
        manifest = pd.read_csv(input_path)
        if "pdb_path" not in manifest.columns:
            raise ValueError("Manifest must contain a pdb_path column.")
        manifest = manifest.copy()
        manifest["pdb_path"] = [str(resolve_path(p, input_path.parent)) for p in manifest["pdb_path"]]
        if "antibody_id" not in manifest.columns:
            manifest["antibody_id"] = [Path(p).stem for p in manifest["pdb_path"]]
        if "pdb_id" not in manifest.columns:
            manifest["pdb_id"] = manifest["antibody_id"]
        if "heavy_chain" not in manifest.columns:
            manifest["heavy_chain"] = HEAVY_CHAIN
        if "light_chain" not in manifest.columns:
            manifest["light_chain"] = LIGHT_CHAIN
        return manifest

    raise ValueError("INPUT_MODE must be 'pdb_file', 'pdb_directory', or 'manifest'.")

input_table = build_input_table(INPUT_MODE, INPUT_PATH)
display(input_table)


## Step 4: run prediction

This cell computes structure descriptors and then scores aggregation risk.


In [ ]:
descriptor_rows = []
error_rows = []

for item in input_table.to_dict(orient="records"):
    pdb_path = Path(item["pdb_path"])
    heavy = str(item.get("heavy_chain", "") or "").strip()
    light = str(item.get("light_chain", "") or "").strip()

    try:
        if not heavy or not light or heavy.lower() == "nan" or light.lower() == "nan":
            heavy, light = infer_two_largest_protein_chains(pdb_path)

        descriptors = compute_surface_descriptors(
            pdb_path,
            heavy_chain=heavy,
            light_chain=light,
            max_curvature_points=MAX_CURVATURE_POINTS,
        )
        descriptors.update({
            "antibody_id": item.get("antibody_id", pdb_path.stem),
            "pdb_id": item.get("pdb_id", pdb_path.stem),
            "source_pdb_path": str(pdb_path),
        })
        if "label" in item:
            descriptors["label"] = item["label"]
        descriptor_rows.append(descriptors)
    except Exception as exc:
        error_rows.append({
            "antibody_id": item.get("antibody_id", pdb_path.stem),
            "pdb_path": str(pdb_path),
            "error": str(exc),
        })

if error_rows:
    errors = pd.DataFrame(error_rows)
    error_path = OUTPUT_DIR / f"{RUN_NAME}_errors.csv"
    errors.to_csv(error_path, index=False)
    display(Markdown(f"Some structures failed. Error file: `{error_path}`"))
    display(errors)

if not descriptor_rows:
    raise ValueError("No structures were successfully scored. Check PDB path and chain IDs.")

descriptors = pd.DataFrame(descriptor_rows)
results = add_batch_scores(descriptors)
results = results.sort_values("thermodynamic_risk_score", ascending=False).reset_index(drop=True)

score_path = OUTPUT_DIR / f"{RUN_NAME}_scores.csv"
results.to_csv(score_path, index=False)
print("Score file:", score_path)


## Step 5: read the result

- `thermodynamic_risk_score`: primary ranking score. Higher means higher predicted aggregation risk.
- `risk_rank`: qualitative risk tier.
- `thermo_call`: thresholded call; `1 = predicted non-pass/high risk`, `0 = predicted pass/lower risk`.


In [ ]:
display_cols = [
    "antibody_id",
    "pdb_id",
    "thermodynamic_risk_score",
    "risk_rank",
    "thermo_call",
    "DeltaMu_proxy",
    "gamma_proxy",
    "DeltaG_star_kT",
    "n_star",
]
display(results[[c for c in display_cols if c in results.columns]])

risk_summary = results["risk_rank"].value_counts().rename_axis("risk_rank").reset_index(name="count")
display(risk_summary)


## Optional: compare to labels

If your manifest includes `label` where `1 = aggregation/non-pass` and `0 = pass`, this cell calculates a confusion matrix. If not, it is skipped.


In [ ]:
if "label" in results.columns:
    y = pd.to_numeric(results["label"], errors="coerce")
    pred = pd.to_numeric(results["thermo_call"], errors="coerce")
    valid = y.notna() & pred.notna()
    y = y[valid].astype(int)
    pred = pred[valid].astype(int)

    tp = int(((y == 1) & (pred == 1)).sum())
    tn = int(((y == 0) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    n = len(y)
    sensitivity = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0

    display(pd.DataFrame([{
        "n": n,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "accuracy": (tp + tn) / n if n else None,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "balanced_accuracy": 0.5 * (sensitivity + specificity),
    }]))
else:
    display(Markdown("No labels provided; prediction-only mode complete."))


## Practical caveat

This is a fast thermodynamic screening model. It should be used to rank antibodies for follow-up, not as a final CMC decision. Results depend on PDB quality, chain selection, missing loops, bound antigen, and formulation context.
